In [1]:
import random

# 1. Настройка графа на основе вашей матрицы
# Узлы: A=0, B=1, C=2, D=3, E=4, F=5, G=6
nodes = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
INF = float('inf')

# Матрица расстояний
dist_matrix = [
    [INF, 3, 14, INF, 6, INF, INF],   # A
    [3, INF, INF, 13, INF, INF, 35],  # B
    [14, INF, INF, INF, INF, INF, INF], # C
    [INF, 13, INF, INF, 16, 8, 6],    # D
    [6, INF, INF, 16, INF, 9, INF],   # E
    [INF, INF, INF, 8, 9, INF, 21],   # F
    [INF, 35, INF, 6, INF, 21, INF]   # G
]

In [ ]:
# 2. Параметры алгоритма
ALPHA = 1.0      # Влияние феромона
BETA = 2.0       # Влияние близости (жадность)
RHO = 0.1        # Коэффициент испарения
Q = 10.0         # Количество феромона для распределения
NUM_ANTS = 10    # Количество муравьев в одной итерации
ITERATIONS = 50  # Количество циклов симуляции


In [3]:
# Инициализация феромонов (небольшое значение на всех существующих путях)
pheromones = [[0.1 if dist_matrix[i][j] != INF else 0 for j in range(len(nodes))] for i in range(len(nodes))]

def get_probabilities(current_node, visited):
    """Вычисляет вероятности перехода в следующий узел"""
    probs = []
    total = 0
    
    for next_node, dist in enumerate(dist_matrix[current_node]):
        if dist != INF and next_node not in visited:
            # Формула ACO: (феромон ^ alpha) * (видимость ^ beta)
            eta = 1.0 / dist  # Видимость = 1 / расстояние
            tau = pheromones[current_node][next_node]
            
            p = (tau ** ALPHA) * (eta ** BETA)
            probs.append([next_node, p])
            total += p
            
    if total == 0: return []
    
    # Нормализация вероятностей
    return [[node, p / total] for node, p in probs]

In [ ]:
# 3. Основной цикл симуляции
start_node = 0  # A
end_node = 4    # E

best_path = None
best_dist = INF

for i in range(ITERATIONS):
    all_paths = []
    
    for ant in range(NUM_ANTS):
        current = start_node
        path = [current]
        visited = {current}
        total_len = 0
        
        # Муравей идет до точки E
        while current != end_node:
            probs = get_probabilities(current, visited)
            if not probs: break # Тупик
            
            next_node = random.choices([p[0] for p in probs], weights=[p[1] for p in probs])[0]
            
            total_len += dist_matrix[current][next_node]
            current = next_node
            path.append(current)
            visited.add(current)
            
        if current == end_node:
            all_paths.append((path, total_len))
            if total_len < best_dist:
                best_dist = total_len
                best_path = path

    # Испарение феромонов
    for r in range(len(nodes)):
        for c in range(len(nodes)):
            pheromones[r][c] *= (1 - RHO)

    # Нанесение новых феромонов
    for path, length in all_paths:
        for j in range(len(path) - 1):
            u, v = path[j], path[j+1]
            pheromones[u][v] += Q / length

# 4. Вывод результата
path_names = " -> ".join([nodes[n] for n in best_path])
print(f"Лучший найденный путь: {path_names}")
print(f"Длина пути: {best_dist}")

Лучший найденный путь: A -> E
Длина пути: 6
